In [6]:

import os
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor
)
from sklearn.base import clone
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
os.makedirs("results", exist_ok=True)

RANDOM_STATE = 42

# --------------------------------------------------
# 1. LOAD DATA
# --------------------------------------------------


In [7]:
data_train = pd.read_csv("data_train.csv")
target_train = pd.read_csv("target_train.csv")

data_test = pd.read_csv("data_test_small.csv")
target_test = pd.read_csv("target_test_small.csv")

print("Training sensors:", data_train.shape)
print("Training targets:", target_train.shape)
print("Test sensors:", data_test.shape)
print("Test targets:", target_test.shape)

Training sensors: (319546, 17)
Training targets: (1773, 2)
Test sensors: (39943, 17)
Test targets: (318, 2)


# --------------------------------------------------
# 2. DATA PREPROCESSING
# --------------------------------------------------

In [8]:




def prepare_sensors(df):

    df = df.copy()

    df["datetime"] = pd.to_datetime(df["datetime"])

    # Average duplicate timestamps
    df = df.groupby("datetime").mean(numeric_only=True)

    df = df.sort_index()

    # Create a continuous minute-level timeline
    full_index = pd.date_range(
        start=df.index.min(),
        end=df.index.max(),
        freq="min"
    )

    df = df.reindex(full_index)
    df.index.name = "datetime"

    # Fill only short interruptions
    df = df.ffill(limit=3)

    return df


sensor_train = prepare_sensors(data_train)
sensor_test = prepare_sensors(data_test)




# --------------------------------------------------
# 3. EXPLORATORY DATA ANALYSIS
# --------------------------------------------------

In [9]:


print("\nMissing values:")
print(sensor_train.isna().mean().sort_values(ascending=False))

print("\nTarget statistics:")
print(target_train["target"].describe())

# Target distribution
plt.figure(figsize=(9, 5))
sns.histplot(target_train["target"], kde=True)
plt.title("Target Distribution")
plt.xlabel("Target concentration")
plt.tight_layout()
plt.savefig("results/target_distribution.png")
plt.close()

# Sensor correlation
plt.figure(figsize=(12, 9))
sns.heatmap(
    sensor_train.corr(),
    cmap="coolwarm",
    center=0
)
plt.title("Sensor Correlation Matrix")
plt.tight_layout()
plt.savefig("results/sensor_correlation.png")
plt.close()





Missing values:
telemetry_12    0.973622
telemetry_13    0.973622
telemetry_14    0.973622
telemetry_15    0.973622
telemetry_4     0.000358
telemetry_0     0.000000
telemetry_2     0.000000
telemetry_1     0.000000
telemetry_7     0.000000
telemetry_6     0.000000
telemetry_5     0.000000
telemetry_3     0.000000
telemetry_11    0.000000
telemetry_10    0.000000
telemetry_9     0.000000
telemetry_8     0.000000
dtype: float64

Target statistics:
count    1773.000000
mean        0.254272
std         0.092208
min         0.000000
25%         0.175258
50%         0.247423
75%         0.314433
max         0.716495
Name: target, dtype: float64


# --------------------------------------------------
# 4. FEATURE ENGINEERING
# --------------------------------------------------

In [10]:


def create_features(target_df, sensors, delay=12):

    target_df = target_df.copy()

    target_df["Дата"] = pd.to_datetime(target_df["Дата"])

    target_df = target_df.rename(
        columns={"Дата": "target_time"}
    )

    # Assume target timestamp is the lab-result time.
    # Use telemetry from 12 minutes earlier.
    feature_times = (
        target_df["target_time"]
        - pd.Timedelta(minutes=delay)
    )

    sensor_columns = list(sensors.columns)

    features = pd.DataFrame(index=target_df.index)

    features["target_time"] = target_df["target_time"]

    # Lag features
    for lag in [0, 5, 10, 15, 30]:

        query_times = (
            feature_times
            - pd.Timedelta(minutes=lag)
        )

        values = sensors.reindex(
            pd.DatetimeIndex(query_times)
        )

        values.index = target_df.index

        values.columns = [
            f"{col}_lag_{lag}"
            for col in sensor_columns
        ]

        features = pd.concat(
            [features, values],
            axis=1
        )

    # Rolling statistics using only past/current data
    for window in [5, 15, 30]:

        rolling_rows = []

        for timestamp in feature_times:

            history = sensors.loc[:timestamp].tail(window)

            row = {}

            for col in sensor_columns:

                series = history[col]

                row[f"{col}_mean_{window}"] = series.mean()
                row[f"{col}_std_{window}"] = series.std()
                row[f"{col}_min_{window}"] = series.min()
                row[f"{col}_max_{window}"] = series.max()

            rolling_rows.append(row)

        rolling_df = pd.DataFrame(
            rolling_rows,
            index=target_df.index
        )

        features = pd.concat(
            [features, rolling_df],
            axis=1
        )

    # Derivatives: difference between current and previous readings
    for col in sensor_columns:

        current = features[f"{col}_lag_0"]
        previous = features[f"{col}_lag_5"]

        features[f"{col}_change_5min"] = current - previous

        features[f"{col}_rate_5min"] = (
            current - previous
        ) / 5.0

    # Time features
    hour = (
        feature_times.dt.hour
        + feature_times.dt.minute / 60
    )

    day = feature_times.dt.dayofweek

    features["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    features["hour_cos"] = np.cos(2 * np.pi * hour / 24)

    features["day_sin"] = np.sin(2 * np.pi * day / 7)
    features["day_cos"] = np.cos(2 * np.pi * day / 7)

    features["target"] = target_df["target"].values

    return features


# 12 minutes is the midpoint of the stated 10–15 minute delay.
train_features = create_features(
    target_train,
    sensor_train,
    delay=12
)

test_features = create_features(
    target_test,
    sensor_test,
    delay=12
)

# Prepare X and y
excluded = ["target", "target_time"]

feature_columns = [
    col for col in train_features.columns
    if col not in excluded
]

X_train = train_features[feature_columns]
y_train = train_features["target"].astype(float)

X_test = test_features[feature_columns]
y_test = test_features["target"].astype(float)

print("\nFeature matrix:", X_train.shape)





Feature matrix: (1773, 308)


# --------------------------------------------------
# 5. CHRONOLOGICAL TRAIN / VALIDATION SPLIT
# --------------------------------------------------

In [11]:


split_index = int(len(X_train) * 0.80)

X_tr = X_train.iloc[:split_index]
X_val = X_train.iloc[split_index:]

y_tr = y_train.iloc[:split_index]
y_val = y_train.iloc[split_index:]




# --------------------------------------------------
# 6. DEFINE MACHINE LEARNING MODELS
# --------------------------------------------------

In [12]:


models = {

    "Ridge": Pipeline([
        ("imputer", SimpleImputer(
            strategy="median",
            add_indicator=True
        )),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=10.0))
    ]),

    "RandomForest": Pipeline([
        ("imputer", SimpleImputer(
            strategy="median",
            add_indicator=True
        )),
        ("model", RandomForestRegressor(
            n_estimators=250,
            min_samples_leaf=3,
            max_features=0.8,
            n_jobs=-1,
            random_state=RANDOM_STATE
        ))
    ]),

    "ExtraTrees": Pipeline([
        ("imputer", SimpleImputer(
            strategy="median",
            add_indicator=True
        )),
        ("model", ExtraTreesRegressor(
            n_estimators=300,
            min_samples_leaf=2,
            max_features=0.9,
            n_jobs=-1,
            random_state=RANDOM_STATE
        ))
    ]),

    "HistGradientBoosting": Pipeline([
        ("imputer", SimpleImputer(
            strategy="median",
            add_indicator=True
        )),
        ("model", HistGradientBoostingRegressor(
            max_iter=250,
            learning_rate=0.05,
            max_leaf_nodes=15,
            l2_regularization=1.0,
            random_state=RANDOM_STATE
        ))
    ])
}




# --------------------------------------------------
# 7. EVALUATION METRICS
# --------------------------------------------------

In [13]:


def calculate_metrics(y_true, y_pred):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mae = mean_absolute_error(y_true, y_pred)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    nonzero = np.abs(y_true) > 1e-8

    mape = (
        np.mean(
            np.abs(
                (y_true[nonzero] - y_pred[nonzero])
                / y_true[nonzero]
            )
        ) * 100
        if nonzero.any()
        else np.nan
    )

    wape = (
        np.sum(np.abs(y_true - y_pred))
        / np.sum(np.abs(y_true))
        * 100
    )

    # Direction accuracy
    if len(y_true) > 1:

        actual_direction = np.sign(np.diff(y_true))
        predicted_direction = np.sign(np.diff(y_pred))

        direction_accuracy = np.mean(
            actual_direction == predicted_direction
        ) * 100

        non_flat = np.abs(np.diff(y_true)) > 1e-8

        direction_nonflat = (
            np.mean(
                actual_direction[non_flat]
                == predicted_direction[non_flat]
            ) * 100
            if non_flat.any()
            else np.nan
        )

    else:
        direction_accuracy = np.nan
        direction_nonflat = np.nan

    return {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE (%)": mape,
        "WAPE (%)": wape,
        "Direction Accuracy (%)": direction_accuracy,
        "Direction Non-flat (%)": direction_nonflat
    }




# --------------------------------------------------
# 8. TRAIN AND VALIDATE MODELS
# --------------------------------------------------

In [14]:


validation_results = {}

for name, model in models.items():

    print(f"\nTraining {name}...")

    model.fit(X_tr, y_tr)

    predictions = model.predict(X_val)

    validation_results[name] = calculate_metrics(
        y_val,
        predictions
    )

validation_df = pd.DataFrame(
    validation_results
).T.sort_values("MAE")

print("\nVALIDATION RESULTS")
print(validation_df)

validation_df.to_csv(
    "results/validation_results.csv"
)

best_model_name = validation_df.index[0]

print("\nBest model:", best_model_name)





Training Ridge...

Training RandomForest...

Training ExtraTrees...

Training HistGradientBoosting...

VALIDATION RESULTS
                           MAE      RMSE   MAPE (%)   WAPE (%)  \
RandomForest          0.068140  0.084046  38.111121  30.164244   
ExtraTrees            0.068148  0.084420  37.446995  30.167733   
HistGradientBoosting  0.070166  0.086654  37.878030  31.060897   
Ridge                 0.078992  0.099212  44.338247  34.968103   

                      Direction Accuracy (%)  Direction Non-flat (%)  
RandomForest                       50.847458               53.731343  
ExtraTrees                         48.022599               50.746269  
HistGradientBoosting               48.870056               51.641791  
Ridge                              48.870056               51.641791  

Best model: RandomForest


# --------------------------------------------------
# 9. FINAL TRAINING AND TEST EVALUATION
# --------------------------------------------------


In [15]:

test_results = {}
fitted_models = {}

for name, model in models.items():

    print(f"\nFitting final {name} model...")

    final_model = clone(model)

    final_model.fit(X_train, y_train)

    fitted_models[name] = final_model

    predictions = final_model.predict(X_test)

    test_results[name] = calculate_metrics(
        y_test,
        predictions
    )

test_df = pd.DataFrame(
    test_results
).T.sort_values("MAE")

print("\nTEST RESULTS")
print(test_df)

test_df.to_csv(
    "results/test_results.csv"
)

best_model = fitted_models[best_model_name]

best_predictions = best_model.predict(X_test)





Fitting final Ridge model...

Fitting final RandomForest model...

Fitting final ExtraTrees model...

Fitting final HistGradientBoosting model...

TEST RESULTS
                           MAE      RMSE   MAPE (%)   WAPE (%)  \
RandomForest          0.062518  0.077729  34.449872  29.812781   
ExtraTrees            0.062791  0.078113  34.734467  29.942853   
HistGradientBoosting  0.064741  0.080934  34.613977  30.872645   
Ridge                 0.067613  0.087238  38.784970  32.242220   

                      Direction Accuracy (%)  Direction Non-flat (%)  
RandomForest                       55.205047               58.139535  
ExtraTrees                         50.473186               53.156146  
HistGradientBoosting               51.735016               54.485050  
Ridge                              55.520505               58.471761  


# --------------------------------------------------
# 10. SAVE MODEL AND PREDICTIONS
# --------------------------------------------------

In [16]:


joblib.dump(
    best_model,
    "results/best_model.pkl"
)

prediction_df = pd.DataFrame({
    "actual": y_test.values,
    "predicted": best_predictions,
    "residual": y_test.values - best_predictions
})

prediction_df.to_csv(
    "results/test_predictions.csv",
    index=False
)




# --------------------------------------------------
# 11. ACTUAL VS PREDICTED PLOT
# --------------------------------------------------

In [17]:


plt.figure(figsize=(14, 6))

plt.plot(
    y_test.values,
    label="Actual",
    linewidth=1
)

plt.plot(
    best_predictions,
    label="Predicted",
    linewidth=1
)

plt.title("Actual vs Predicted Target Concentration")
plt.xlabel("Test observation")
plt.ylabel("Target concentration")
plt.legend()
plt.tight_layout()

plt.savefig(
    "results/actual_vs_predicted.png"
)

plt.close()




# --------------------------------------------------
# 12. RESIDUAL ANALYSIS
# --------------------------------------------------

In [18]:


residuals = y_test.values - best_predictions

plt.figure(figsize=(9, 5))

sns.histplot(residuals, kde=True)

plt.title("Residual Distribution")
plt.xlabel("Residual")

plt.tight_layout()

plt.savefig(
    "results/residual_distribution.png"
)

plt.close()


plt.figure(figsize=(10, 5))

plt.scatter(
    best_predictions,
    residuals,
    alpha=0.5
)

plt.axhline(0, linestyle="--")

plt.title("Residuals vs Predictions")
plt.xlabel("Predicted values")
plt.ylabel("Residuals")

plt.tight_layout()

plt.savefig(
    "results/residuals_vs_predictions.png"
)

plt.close()




# --------------------------------------------------
# 13. PERMUTATION FEATURE IMPORTANCE
# --------------------------------------------------

In [19]:


print("\nCalculating feature importance...")

sample_size = min(250, len(X_test))

rng = np.random.RandomState(RANDOM_STATE)

sample_indices = np.sort(
    rng.choice(
        len(X_test),
        size=sample_size,
        replace=False
    )
)

importance = permutation_importance(
    best_model,
    X_test.iloc[sample_indices],
    y_test.iloc[sample_indices],
    n_repeats=8,
    random_state=RANDOM_STATE,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "feature": feature_columns,
    "importance": importance.importances_mean,
    "std": importance.importances_std
}).sort_values(
    "importance",
    ascending=False
)

importance_df.to_csv(
    "results/feature_importance.csv",
    index=False
)

plt.figure(figsize=(10, 8))

top_features = importance_df.head(15).sort_values(
    "importance"
)

plt.barh(
    top_features["feature"],
    top_features["importance"]
)

plt.title("Top 15 Important Features")
plt.xlabel("Permutation importance")

plt.tight_layout()

plt.savefig(
    "results/feature_importance.png"
)

plt.close()





Calculating feature importance...


# --------------------------------------------------
# 14. RESIDUAL AUTOCORRELATION
# --------------------------------------------------

In [20]:


from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.stattools import durbin_watson

print("\nDurbin-Watson statistic:")
print(durbin_watson(residuals))

print("\nLjung-Box test:")

print(
    acorr_ljungbox(
        residuals,
        lags=[5, 10, 20],
        return_df=True
    )
)





Durbin-Watson statistic:
1.0731590083513187

Ljung-Box test:
       lb_stat     lb_pvalue
5   134.314369  2.888494e-27
10  152.185195  1.323397e-27
20  160.551907  5.874887e-24


# --------------------------------------------------
# 15. FINAL SUMMARY
# --------------------------------------------------

In [21]:


print("\n===================================")
print("PROJECT COMPLETED")
print("===================================")

print("Best model:", best_model_name)

print("\nBest model test metrics:")
print(test_df.loc[best_model_name])

print("\nFiles saved in results/ directory:")
print("- validation_results.csv")
print("- test_results.csv")
print("- test_predictions.csv")
print("- best_model.pkl")
print("- feature_importance.csv")
print("- target_distribution.png")
print("- sensor_correlation.png")
print("- actual_vs_predicted.png")
print("- residual_distribution.png")
print("- residuals_vs_predictions.png")
print("- feature_importance.png")



PROJECT COMPLETED
Best model: RandomForest

Best model test metrics:
MAE                        0.062518
RMSE                       0.077729
MAPE (%)                  34.449872
WAPE (%)                  29.812781
Direction Accuracy (%)    55.205047
Direction Non-flat (%)    58.139535
Name: RandomForest, dtype: float64

Files saved in results/ directory:
- validation_results.csv
- test_results.csv
- test_predictions.csv
- best_model.pkl
- feature_importance.csv
- target_distribution.png
- sensor_correlation.png
- actual_vs_predicted.png
- residual_distribution.png
- residuals_vs_predictions.png
- feature_importance.png
